# Parte 2 — Evaluación con golden set

**Nombres: Jessica Ballesteros, Miguel Álvarez, Darlyn Ludeña**

Este notebook ejecuta la evaluación definida en `evaluation.py` sobre `golden_set.json`. Usa los seis papers en inglés de `corpus/`, embeddings `bge-m3` en la H200 (`embed_local_multilingue`, verificado el 2026-08-27) y Qdrant en memoria. Para generación sin clave, se conecta Ollama a la H200 con `qwen3:32b` (fila `open_weight_pequeno` configurada en la tabla del repositorio). La pregunta adversarial se considera respondible porque tiene una fuente. Las dos preguntas negativas carecen de fuente y se evalúan por abstención.

## Preparación

Conecta GlobalProtect antes de ejecutar. El notebook usa Qdrant en memoria, por lo que no hace falta Docker para esta evaluación. Sin clave, envía generación a Ollama en la H200. Si tu grupo va a usar un generador distinto, configura `OPENAI_API_KEY` en `.env` o ajusta `OLLAMA_URL` y el modelo antes de correr. La evaluación genera 20 respuestas (10 preguntas para cada k) y la inspección 2.c genera 3 respuestas adicionales.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd

raiz = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(raiz))
os.chdir(raiz)
os.environ.setdefault('EMBEDDING_BACKEND', 'h200')
os.environ.setdefault('QDRANT_URL', ':memory:')
os.environ.setdefault('OLLAMA_URL', 'http://172.28.230.10:11434')

print('Proyecto:', raiz)
print('Backend embeddings:', os.environ['EMBEDDING_BACKEND'])
print('Qdrant:', os.environ['QDRANT_URL'])
print('Ollama:', os.environ['OLLAMA_URL'])
print('Golden set:', (raiz / 'golden_set.json').is_file())
print('Corpus:', len(list((raiz / 'corpus').glob('*.pdf'))), 'PDFs')

In [ ]:
from evaluation import load_golden_set, evaluate_retrieval, escribir_csv, acierta
from rag_pipeline import RagPipeline, EMBEDDING_MODEL, EMBEDDING_BACKEND

golden = load_golden_set(raiz / 'golden_set.json')
assert len(golden) == 10, f'Se esperaban 10 preguntas y hay {len(golden)}'
assert sum(not q['documentos_fuente'] for q in golden) >= 2, 'Faltan preguntas negativas'
print('Preguntas:', len(golden))
print('Modelo embeddings:', EMBEDDING_MODEL)
print('Backend:', EMBEDDING_BACKEND)

## 2.b — Recuperación y abstención para `k=3` y `k=5`

Para cada `k`, el pipeline vuelve a indexar el corpus en Qdrant, calcula Hit Rate y MRR solo sobre las ocho consultas respondibles, y genera respuestas para medir abstención correcta e indebida. El CSV conserva una fila por consulta y constituye el resultado crudo.

In [ ]:
pipeline = RagPipeline()
resumenes = []
filas = {}
carpeta_resultados = raiz / 'resultados'
carpeta_resultados.mkdir(parents=True, exist_ok=True)

for k in (3, 5):
    print(f'\n--- Evaluación k={k} ---')
    chunks = pipeline.ingest(raiz / 'corpus')
    print('Fragmentos ingeridos:', len(chunks))
    pipeline.index(chunks)
    reporte = evaluate_retrieval(golden, pipeline, k=k, generar=True)
    salida_csv = carpeta_resultados / f'parte2_k{k}.csv'
    escribir_csv(reporte['rows'], salida_csv, EMBEDDING_MODEL)
    filas[k] = reporte['rows']
    resumenes.append({
        'k': k, 'n_preguntas': reporte['n_preguntas'],
        'n_respondibles': reporte['n_respondibles'], 'n_negativas': reporte['n_negativas'],
        'hit_rate': reporte['hit_rate'], 'mrr': reporte['mrr'],
        'abstencion_correcta': reporte['abstencion_correcta'],
        'abstencion_indebida': reporte['abstencion_indebida'],
        'csv': str(salida_csv.relative_to(raiz))
    })
    print('Hit Rate:', reporte['hit_rate'])
    print('MRR:', reporte['mrr'])
    print('Abstención correcta:', reporte['abstencion_correcta'])
    print('Abstención indebida:', reporte['abstencion_indebida'])
    print('CSV:', salida_csv)

archivo_crudo = carpeta_resultados / 'resultados.csv'
escribir_csv([*filas[3], *filas[5]], archivo_crudo, EMBEDDING_MODEL)
print('CSV crudo combinado:', archivo_crudo, '(20 filas: 10 por cada k)')

## Tabla de resultados para el informe

Hit Rate y MRR usan únicamente las consultas respondibles. Las dos tasas de abstención tienen denominadores separados: negativas y respondibles.

In [ ]:
df_metricas = pd.DataFrame(resumenes)
display(df_metricas)
archivo_resumen = carpeta_resultados / 'parte2_resumen.csv'
df_metricas.to_csv(archivo_resumen, index=False, encoding='utf-8-sig')
print('Resumen guardado en:', archivo_resumen)

### Corrida realizada el 27-09-2026

Evaluación ejecutada con `embed_local_multilingue` (`BAAI/bge-m3`) en la H200 y generación `ollama:qwen3:32b`; Qdrant se ejecutó en memoria. Los valores se derivan de los CSV crudos incluidos en `resultados/`.

| k | Hit Rate (8 respondibles) | MRR | Abstención correcta (2 negativas) | Abstención indebida (8 respondibles) |
|---:|---:|---:|---:|---:|
| 3 | 0.625 | 0.542 | 1.000 | 0.125 |
| 5 | 0.625 | 0.542 | 1.000 | 0.250 |

Al subir de k=3 a k=5 no cambió Hit Rate ni MRR en esta corrida; la abstención indebida aumentó de 0.125 a 0.250. Es un resultado observado en este golden set, no una expectativa general.

## 2.c — Inspección de los tres peores casos

Se ordenan las consultas respondibles de peor a mejor por posición del primer acierto (las que no acertaron quedan primero). Revisa los fragmentos recuperados y las respuestas para clasificar cada fallo como ingesta, fragmentación, recuperación, prompt o generación.

In [ ]:
# k=5: evaluar casos débiles y recuperar texto/puntajes como evidencia.
respondibles = [q for q in golden if q['documentos_fuente']]
candidatos = []
for q in respondibles:
    hits = pipeline.retrieve(q['pregunta'], top_k=5)
    posicion = next((i for i, hit in enumerate(hits, start=1) if acierta(hit, q)), None)
    candidatos.append((posicion or 99, q, hits))
orden = sorted(candidatos, key=lambda x: x[0], reverse=True)
peores_casos = []
for posicion, q, hits in orden[:3]:
    answer = pipeline.answer(q['pregunta'], top_k=5)
    peores_casos.append({
        'id': q['id'], 'pregunta': q['pregunta'],
        'respuesta_esperada': q['respuesta_esperada'],
        'hits': hits, 'respuesta_generada': answer['answer'],
        'generador': answer['generator'], 'abstuvo': answer['abstained'],
        'posicion_primer_acierto': None if posicion == 99 else posicion
    })
    print(f"\nCaso {q['id']}: {q['pregunta']}")
    print('Evidencia esperada:', q['documentos_fuente'], '|', q['fragmento_esperado'])
    for pos, h in enumerate(hits, start=1):
        print(f"  {pos}. score={h['score']:.4f} documento={h['document']} chunk={h['chunk_id']}")
        print('    ', h['text'][:500].replace('\n', ' '))
    print('Respuesta:', answer['answer'])
    print('Generador:', answer['generator'], '| abstuvo:', answer['abstained'])

### Limitación que debe quedar explícita

La pregunta multi-fragmento se marca como acierto si el top-k encuentra un fragmento de cualquiera de sus documentos fuente; eso mide recuperación, no demuestra que la respuesta combine ambas fuentes correctamente. Anota esa limitación al interpretar las métricas. Los resultados generados también dependen del generador disponible en `.env`/Ollama; registra cuál indica la salida.